In [1]:
# Install required libraries (run once)
!pip install opencv-python-headless moviepy -q

# Import libraries we will use
import cv2
import numpy as np
from google.colab import files

print("Libraries installed and ready!")

Libraries installed and ready!


In [14]:
# Upload your video file(s)
uploaded = files.upload()

# Show the names of uploaded files
print("Uploaded files:")
for name in uploaded.keys():
    print(" - " + name)

Saving hairdryer_wind.mp4 to hairdryer_wind.mp4
Uploaded files:
 - hairdryer_wind.mp4


In [4]:
# Show which video was uploaded (quick check)
print("Current uploaded video name:", list(uploaded.keys())[0])

Current uploaded video name: hairdryer_wind4 (1).mp4


In [5]:
# Quick check: show the name of the latest uploaded video
print("Latest uploaded video:", list(uploaded.keys())[0])

Latest uploaded video: hairdryer_wind4 (1).mp4


In [15]:
# Analyze the latest uploaded video
video_filename = list(uploaded.keys())[0]

print("Analyzing video:", video_filename)

cap = cv2.VideoCapture(video_filename)
ret, frame1 = cap.read()

if not ret:
    print("Error: Could not read video")
else:
    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    jet_count = 0
    natural_count = 0
    weak_count = 0
    frame_idx = 0

    while cap.isOpened():
        ret, frame2 = cap.read()
        if not ret:
            break

        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prvs, next_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        avg_mag = np.mean(mag)
        ang_var = np.var(ang)

        if avg_mag > 4 and ang_var < 2.0:
            jet_count += 1
        elif avg_mag > 2.5 and ang_var > 1.8:
            natural_count += 1
        else:
            weak_count += 1

        prvs = next_frame
        frame_idx += 1

    cap.release()

    total_frames = frame_idx
    print(f"Total frames analyzed: {total_frames}")
    print(f"jet_indoor frames: {jet_count} ({jet_count/total_frames*100:.1f}%)")
    print(f"natural_outdoor frames: {natural_count} ({natural_count/total_frames*100:.1f}%)")
    print(f"weak_or_static frames: {weak_count} ({weak_count/total_frames*100:.1f}%)")

Analyzing video: hairdryer_wind.mp4
Total frames analyzed: 966
jet_indoor frames: 2 (0.2%)
natural_outdoor frames: 271 (28.1%)
weak_or_static frames: 693 (71.7%)
